In [1]:
!uv pip install -q deepagents langchain-openai

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [3]:
from deepagents import create_deep_agent
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.backends.utils import create_file_data
from deepagents.backends.state import StateBackend
from langchain.tools import ToolRuntime
from langchain.chat_models import init_chat_model
from langgraph.store.memory import InMemoryStore

# Pre-configure files
initial_files = {
    "/project/README.md": create_file_data("# My Project\n\nInitial documentation."),
    "/project/src/app.py": create_file_data("def main():\n    print('Hello!')")
}

# Create runtime with pre-populated state
runtime = ToolRuntime(
    state={"messages": [], "files": initial_files},
    context=None,
    tool_call_id="tc",
    store=InMemoryStore(),
    stream_writer=lambda _: None,
    config={},
)

# Create backend with pre-configured files
backend = StateBackend(runtime)
model = init_chat_model(model="openai:gpt-4.1-mini")
agent = create_deep_agent(backend=backend, model=model)
input = {
    "messages": [{"role": "user", "content": "List files in project directory."}],
    "files": initial_files,
}
for mode, chunk in agent.stream(
    input,
    stream_mode=["values"],
    config={"configurable": {"thread_id": "openai"}},  # Dual-mode for HITL support
):
    if "messages" in chunk:
        chunk["messages"][-1].pretty_print()

================================ Human Message =================================

List files in project directory.
================================ Human Message =================================

List files in project directory.
================================== Ai Message ==================================
Tool Calls:
  ls (call_vE7ivnaQtzLmwAREO8Q6JGso)
 Call ID: call_vE7ivnaQtzLmwAREO8Q6JGso
  Args:
    path: /
================================= Tool Message =================================
Name: ls

['/project/']
================================== Ai Message ==================================
Tool Calls:
  ls (call_lVoIp9qfQNYSOtXNTm5lRTGD)
 Call ID: call_lVoIp9qfQNYSOtXNTm5lRTGD
  Args:
    path: /project
================================= Tool Message =================================
Name: ls

['/project/README.md', '/project/src/']
================================== Ai Message ==================================

The project directory contains the following files and folders:
-